# Week 10 Companion Notebook: Streaming Data & Real-Time Pipelines

## ISM 6562 — Big Data for Business Applications

---

This notebook accompanies the Week 10 lecture on **Apache Kafka and Spark Structured Streaming**. Building on the batch Transform layer from Week 9, you will now add a **real-time streaming layer** that processes events as they arrive.

You will work through two complete business scenarios:

- **HealthPulse (Lambda Architecture)** — Real-time anomaly detection from IoT medical devices, complementing the nightly batch pipeline from Week 9
- **GreenRoute (Kappa Architecture)** — Live fleet tracking where all data flows through Kafka with no separate batch layer

By the end of this notebook, you will have hands-on experience with:

1. Creating and managing Kafka topics programmatically
2. Producing and consuming events with kafka-python
3. Reading Kafka streams into Spark Structured Streaming DataFrames
4. Applying filters, windowed aggregations, and anomaly rules on streaming data
5. Understanding the code-level similarities between batch and streaming Spark
6. Comparing Lambda and Kappa architecture patterns in practice

---

## Part 0: Setup & Environment Verification

The `docker-compose.yml` for this week creates a streaming pipeline with the following components:

| Component | Container Name | Role |
|---|---|---|
| **Zookeeper** | `zookeeper` | Kafka cluster coordination and metadata management |
| **Kafka** | `kafka` | Message broker — ingests streaming events from producers |
| **Kafka UI** | `kafka-ui` | Browser-based inspector for topics, partitions, messages, and consumer lag |
| **Spark Master** | `spark-master` | Coordinates Structured Streaming jobs |
| **Spark Worker** | `spark-worker` | Executes streaming micro-batch tasks (2 cores, 2 GB RAM) |
| **Jupyter** | `streaming-jupyter` | PySpark notebook with kafka-python (this notebook!) |

**Before running this notebook**, start the cluster from your terminal:

```bash
cd week10-streaming-real-time-pipelines
docker compose up -d
```

Then open this notebook at: **http://localhost:8888?token=spark**

You can monitor the infrastructure at:

- **Spark Master UI**: http://localhost:8080
- **Spark Worker UI**: http://localhost:8081
- **Spark Application UI**: http://localhost:4040 (active while a streaming job runs)
- **Kafka UI**: http://localhost:8088 — open this in a second tab; you will use it throughout Part 1 to watch topics and messages appear as you produce events


In [ ]:
# Create a SparkSession with the Spark-Kafka connector package
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Week10-StreamingPipelines") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .config("spark.executor.memory", "1g") \
    .config("spark.driver.memory", "1g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Application ID: {spark.sparkContext.applicationId}")
print(f"Master: {spark.sparkContext.master}")
print("\nSparkSession created — check http://localhost:8080 to see your application.")

In [ ]:
# Verify Kafka is running and accessible
from kafka import KafkaAdminClient
from kafka.errors import NoBrokerAvailable
import time

max_retries = 5
for attempt in range(max_retries):
    try:
        admin = KafkaAdminClient(bootstrap_servers="kafka:9092", client_id="setup-check")
        topics = admin.list_topics()
        print(f"Kafka is running. Existing topics: {topics}")
        admin.close()
        break
    except NoBrokerAvailable:
        if attempt < max_retries - 1:
            print(f"Kafka not ready yet (attempt {attempt + 1}/{max_retries}), retrying in 5s...")
            time.sleep(5)
        else:
            print("ERROR: Could not connect to Kafka. Make sure 'docker compose up -d' has finished.")

In [ ]:
# Download all Week 10 sample data from the course data repository
import os
import subprocess

REPO = "https://raw.githubusercontent.com/prof-tcsmith/ism6562s26-class/main"
DATA_DIR = "/home/jovyan/data"

files_to_download = [
    # HealthPulse streaming data
    ("week10/data/healthpulse/sample-device-events.json", "healthpulse/sample-device-events.json"),
    ("week10/data/healthpulse/sample-claim-events.json", "healthpulse/sample-claim-events.json"),
    # GreenRoute streaming data
    ("week10/data/greenroute/sample-gps-events.json", "greenroute/sample-gps-events.json"),
    ("week10/data/greenroute/sample-delivery-events.json", "greenroute/sample-delivery-events.json"),
    ("week10/data/greenroute/sample-weather-alerts.json", "greenroute/sample-weather-alerts.json"),
]

downloaded = 0
skipped = 0
for remote_path, local_path in files_to_download:
    full_local = os.path.join(DATA_DIR, local_path)
    os.makedirs(os.path.dirname(full_local), exist_ok=True)
    if os.path.exists(full_local):
        skipped += 1
        continue
    result = subprocess.run(
        ["curl", "-sL", f"{REPO}/{remote_path}", "-o", full_local],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        downloaded += 1
    else:
        print(f"  WARNING: Failed to download {remote_path}")

print(f"Download complete: {downloaded} new files, {skipped} already present.")
print(f"Total files: {downloaded + skipped}")

In [ ]:
# Verify downloaded data files
import os, json

data_files = {
    "healthpulse/sample-device-events.json": "HealthPulse device events",
    "healthpulse/sample-claim-events.json": "HealthPulse claim events",
    "greenroute/sample-gps-events.json": "GreenRoute GPS events",
    "greenroute/sample-delivery-events.json": "GreenRoute delivery events",
    "greenroute/sample-weather-alerts.json": "GreenRoute weather alerts",
}

for path, description in data_files.items():
    full_path = os.path.join(DATA_DIR, path)
    if os.path.exists(full_path):
        with open(full_path) as f:
            data = json.load(f)
        count = len(data) if isinstance(data, list) else 1
        print(f"  {description}: {count:,} records")
    else:
        print(f"  {description}: MISSING — check download step above")

---

## Part 1: Kafka Fundamentals

Before connecting Spark Structured Streaming to Kafka, we need to understand Kafka itself. Kafka is a **distributed event log** — producers write events to topics, and consumers read them. Unlike a traditional message queue, **consuming an event does not delete it**. Events are retained for a configurable period (7 days in our setup), and any consumer can re-read from any offset.

Key concepts:

- **Topic** — A named stream of events (like a database table, but append-only)
- **Partition** — A topic is divided into partitions for parallelism
- **Producer** — Writes events to a topic
- **Consumer** — Reads events from a topic
- **Consumer Group** — A set of consumers that share the work of reading a topic
- **Offset** — The position of a consumer within a partition

### 1a. Creating Topics

While our Kafka broker has `auto.create.topics.enable=true` (topics are created automatically on first produce), it is best practice to create topics explicitly so you can control the partition count and replication factor.

In [ ]:
# Create Kafka topics programmatically
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

admin = KafkaAdminClient(bootstrap_servers="kafka:9092", client_id="topic-creator")

topics_to_create = [
    NewTopic(name="test-topic", num_partitions=1, replication_factor=1),
    NewTopic(name="device-events", num_partitions=3, replication_factor=1),
    NewTopic(name="gps-stream", num_partitions=3, replication_factor=1),
    NewTopic(name="delivery-events", num_partitions=2, replication_factor=1),
    NewTopic(name="weather-alerts", num_partitions=1, replication_factor=1),
    NewTopic(name="critical-alerts", num_partitions=2, replication_factor=1),
]

for topic in topics_to_create:
    try:
        admin.create_topics([topic])
        print(f"  Created topic: {topic.name} ({topic.num_partitions} partitions)")
    except TopicAlreadyExistsError:
        print(f"  Topic already exists: {topic.name}")

admin.close()

In [ ]:
# List all topics to verify
admin = KafkaAdminClient(bootstrap_servers="kafka:9092", client_id="topic-lister")
topics = admin.list_topics()
print("Current Kafka topics:")
for t in sorted(topics):
    if not t.startswith("_"):  # Skip internal topics
        print(f"  - {t}")
admin.close()

::: {.callout-tip}
## Watch this in Kafka UI

Switch to your Kafka UI tab at [http://localhost:8088](http://localhost:8088) and click **Topics**. You should see all five topics from the previous cell (`test-topic`, `device-events`, `gps-stream`, `delivery-events`, `weather-alerts`) listed with 0 messages so far. As the next cells produce events, the message counts in this view will climb — refresh to confirm each produce actually landed in Kafka.
:::


### 1b. Producing Events

A **KafkaProducer** serializes data and writes it to a topic. Each event has a **key** (optional, used for partitioning) and a **value** (the payload). We serialize our events as JSON.

In [ ]:
# Create a KafkaProducer with JSON serialization
from kafka import KafkaProducer
import json

producer = KafkaProducer(
    bootstrap_servers="kafka:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: k.encode("utf-8") if k else None,
)

# Produce 10 sample device events manually
sample_events = [
    {"device_id": f"dev-{i:03d}", "device_type": "heart_monitor", "patient_id": f"P{1000+i}",
     "hospital_id": "H01", "timestamp": f"2026-04-08T10:{i:02d}:00Z",
     "reading": {"heart_rate": 72 + i * 5, "spo2": 98 - i, "bp_systolic": 120 + i * 3}}
    for i in range(10)
]

for event in sample_events:
    producer.send("test-topic", key=event["device_id"], value=event)

producer.flush()
print(f"Produced {len(sample_events)} events to 'test-topic'")
print(f"\nSample event:\n{json.dumps(sample_events[0], indent=2)}")

In [ ]:
# Load sample data file and produce all events to Kafka
import json

with open("/home/jovyan/data/healthpulse/sample-device-events.json") as f:
    device_events = json.load(f)

print(f"Loaded {len(device_events)} device events from file")
print(f"Sample event keys: {list(device_events[0].keys())}")

# Produce each event to the device-events topic
for event in device_events:
    key = event.get("device_id", "unknown")
    producer.send("device-events", key=key, value=event)

producer.flush()
print(f"\nProduced {len(device_events)} events to 'device-events' topic")

### 1c. Consuming Events

A **KafkaConsumer** reads events from a topic. Key behaviors:

- Each consumer tracks its **offset** (position) in each partition
- Consumers in the same **group** share partitions — each partition is read by exactly one consumer in the group
- Consumers in different groups each get all events independently

The `auto_offset_reset="earliest"` setting tells the consumer to start from the beginning of the topic if it has no prior offset.

In [ ]:
# Create a KafkaConsumer and read events from test-topic
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    "test-topic",
    bootstrap_servers="kafka:9092",
    auto_offset_reset="earliest",
    group_id="notebook-consumer-group-1",
    value_deserializer=lambda m: json.loads(m.decode("utf-8")),
    consumer_timeout_ms=5000,  # Stop after 5 seconds of no new messages
)

print("Reading events from 'test-topic':\n")
count = 0
for message in consumer:
    count += 1
    if count <= 5:  # Show first 5 in detail
        print(f"  Partition: {message.partition} | Offset: {message.offset} | "
              f"Key: {message.key.decode('utf-8') if message.key else None}")
        print(f"    Value: {json.dumps(message.value)[:100]}...")
    elif count == 6:
        print(f"  ... (showing remaining count only)")

print(f"\nTotal events consumed: {count}")
consumer.close()

**Key Observation:** Notice the `partition` and `offset` values. Events with the same key always go to the same partition (Kafka hashes the key). The offset is a monotonically increasing integer — it is how Kafka tracks what you have read.

### 1d. Kafka Concepts Demo

Let us explore three fundamental Kafka behaviors:

1. **Partitioning** — Events distribute across partitions based on their key
2. **Consumer groups** — Multiple consumers in a group share the workload
3. **Durability** — Consuming does NOT delete events; you can re-read from any offset

In [ ]:
# DEMO 1: Partition distribution
# The 'device-events' topic has 3 partitions. Let's see how events distributed.

consumer = KafkaConsumer(
    "device-events",
    bootstrap_servers="kafka:9092",
    auto_offset_reset="earliest",
    group_id="partition-demo-group",
    value_deserializer=lambda m: json.loads(m.decode("utf-8")),
    consumer_timeout_ms=5000,
)

partition_counts = {}
for message in consumer:
    p = message.partition
    partition_counts[p] = partition_counts.get(p, 0) + 1

consumer.close()

print("Event distribution across partitions in 'device-events':")
for p in sorted(partition_counts):
    bar = "█" * (partition_counts[p] // 10)
    print(f"  Partition {p}: {partition_counts[p]:>5} events  {bar}")
print(f"\nTotal: {sum(partition_counts.values())} events")
print("\nEvents with the same key (device_id) always land in the same partition.")

In [ ]:
# DEMO 2: Re-reading from offset 0 — consuming does NOT delete events
# We already consumed all events above. Now let's read them again from the start.

consumer2 = KafkaConsumer(
    "device-events",
    bootstrap_servers="kafka:9092",
    auto_offset_reset="earliest",
    group_id="re-read-demo-group",  # New group = starts from beginning
    value_deserializer=lambda m: json.loads(m.decode("utf-8")),
    consumer_timeout_ms=5000,
)

reread_count = sum(1 for _ in consumer2)
consumer2.close()

print(f"Re-read {reread_count} events from 'device-events' using a new consumer group.")
print("\nKey insight: Kafka is a LOG, not a QUEUE.")
print("Consuming does not remove events. Any consumer group can re-read from any offset.")
print("This is what makes Kafka ideal for both real-time processing AND historical replay.")

---

## Part 2: Spark Structured Streaming Basics

Spark Structured Streaming treats a live data stream as an **unbounded table** that grows as new events arrive. The API is nearly identical to batch DataFrames — you use the same `select`, `filter`, `groupBy`, and `join` operations. The key differences:

| | Batch | Streaming |
|---|---|---|
| **Read** | `spark.read` | `spark.readStream` |
| **Write** | `df.write` | `df.writeStream` |
| **Execution** | Runs once, produces final result | Runs continuously, produces incremental results |
| **Output modes** | N/A | `append`, `update`, `complete` |

### 2a. Reading from Kafka

To read a Kafka topic as a streaming DataFrame, use `spark.readStream.format("kafka")`. The resulting DataFrame has a fixed schema with columns like `key`, `value`, `topic`, `partition`, `offset`, and `timestamp`. The `value` column contains the raw event bytes — we need to cast it to string and parse the JSON.

In [ ]:
# Read device-events topic as a streaming DataFrame
raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "device-events")
    .option("startingOffsets", "earliest")
    .load()
)

# Inspect the raw schema — Kafka gives us fixed columns
print("Raw Kafka stream schema:")
raw_stream.printSchema()

In [ ]:
# Parse the value column: binary -> string -> JSON struct
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, TimestampType

# Define the schema for our device events
# (Adjust field names/types based on the actual data format)
device_schema = StructType([
    StructField("device_id", StringType()),
    StructField("device_type", StringType()),
    StructField("patient_id", StringType()),
    StructField("hospital_id", StringType()),
    StructField("timestamp", StringType()),
    StructField("reading", StructType([
        StructField("heart_rate", FloatType()),
        StructField("spo2", FloatType()),
        StructField("bp_systolic", FloatType()),
        StructField("bp_diastolic", FloatType()),
        StructField("temperature", FloatType()),
    ])),
])

parsed_stream = (
    raw_stream
    .selectExpr("CAST(value AS STRING) as json_str", "timestamp as kafka_timestamp")
    .select(
        F.from_json(F.col("json_str"), device_schema).alias("data"),
        F.col("kafka_timestamp"),
    )
    .select("data.*", "kafka_timestamp")
)

print("Parsed stream schema:")
parsed_stream.printSchema()

### 2b. Basic Streaming Query

A streaming query starts when you call `.writeStream.start()`. It runs continuously until you stop it. We use `outputMode("append")` because our filter produces new rows without modifying old ones.

**Important:** In a notebook, we use `awaitTermination(timeout=...)` to let the query run for a fixed time, then stop it. In production, streaming queries run indefinitely.

In [ ]:
# Filter to anomalous readings only and write to console
anomaly_stream = parsed_stream.filter(
    (F.col("reading.heart_rate") > 100) |
    (F.col("reading.spo2") < 92) |
    (F.col("reading.bp_systolic") > 160)
)

query = (
    anomaly_stream
    .select("device_id", "patient_id", "hospital_id", "timestamp",
            "reading.heart_rate", "reading.spo2", "reading.bp_systolic")
    .writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", False)
    .option("numRows", 20)
    .trigger(processingTime="5 seconds")
    .start()
)

print("Streaming query started — anomalous readings will appear below.")
print("The query processes micro-batches every 5 seconds.\n")

# Let the query run for 30 seconds, then stop
query.awaitTermination(timeout=30)
query.stop()
print("\nQuery stopped.")

### 2c. Windowed Aggregation

Windowed aggregations group events by a **time window** and compute aggregates within each window. This is one of the most powerful streaming patterns.

- **Tumbling window** — Fixed-size, non-overlapping intervals (e.g., every 1 minute)
- **Sliding window** — Overlapping intervals (e.g., 5-minute window sliding every 1 minute)

We use `outputMode("complete")` because the aggregation produces a full result table that updates as new events arrive.

In [ ]:
# Count events per device_type in 1-minute tumbling windows
windowed_counts = (
    parsed_stream
    .withColumn("event_time", F.to_timestamp("timestamp"))
    .groupBy(
        F.window("event_time", "1 minute"),
        "device_type"
    )
    .count()
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "device_type",
        "count"
    )
)

query_windowed = (
    windowed_counts
    .writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", False)
    .option("numRows", 30)
    .trigger(processingTime="5 seconds")
    .start()
)

print("Windowed aggregation running — counts per device_type per minute window:\n")
query_windowed.awaitTermination(timeout=30)
query_windowed.stop()
print("\nWindowed query stopped.")

### 2d. Streaming vs Batch — Side-by-Side Comparison

One of Spark Structured Streaming's greatest strengths is that the code looks almost identical to batch. Let us compare reading the same Kafka data as batch vs. stream.

**Batch approach** (using `spark.read`):

In [ ]:
# BATCH: Read all current data from Kafka as a static DataFrame
batch_df = (
    spark.read
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "device-events")
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load()
    .selectExpr("CAST(value AS STRING) as json_str")
    .select(F.from_json(F.col("json_str"), device_schema).alias("data"))
    .select("data.*")
)

print("BATCH read — all events at once:")
print(f"  Total rows: {batch_df.count()}")
batch_df.groupBy("device_type").count().show()

**Streaming approach** (using `spark.readStream`) — conceptually identical:

```python
# STREAMING: Same logic, but readStream instead of read
stream_df = (
    spark.readStream                    # <-- only change: read -> readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "device-events")
    .option("startingOffsets", "earliest")
    .load()
    .selectExpr("CAST(value AS STRING) as json_str")
    .select(from_json(col("json_str"), device_schema).alias("data"))
    .select("data.*")
)

# Same groupBy, but writeStream instead of show()
stream_df.groupBy("device_type").count() \
    .writeStream.outputMode("complete").format("console").start()
```

**Key Takeaway:** The transformation logic (`select`, `filter`, `groupBy`) is identical. Only the read/write entry points change. This means your batch business rules transfer directly to streaming.

---

## Part 3: HealthPulse — Real-Time Anomaly Detection (Lambda Architecture)

In Week 9, HealthPulse detected anomalies in a **nightly batch** pipeline — reading all accumulated device readings from HDFS, running threshold rules, and writing flagged records to Parquet. That approach works, but a patient with a dangerously high heart rate at 2 AM would not be flagged until the batch runs the next morning.

Now we add a **real-time streaming layer** that detects the same anomalies within seconds. This is the **Lambda Architecture** pattern:

- **Batch layer** (Week 9): Comprehensive analysis on all historical data, runs nightly
- **Speed layer** (Week 10): Real-time anomaly detection on live events, fires alerts immediately
- **Serving layer**: Both feed into a unified view for clinicians

### Step 3.1: Load Sample Events into Kafka

We already produced the device events in Part 1b. Let us verify they are in the topic and also load the claim events.

In [ ]:
# Produce claim events to Kafka
with open("/home/jovyan/data/healthpulse/sample-claim-events.json") as f:
    claim_events = json.load(f)

claim_producer = KafkaProducer(
    bootstrap_servers="kafka:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: k.encode("utf-8") if k else None,
)

for event in claim_events:
    key = event.get("claim_id", event.get("patient_id", "unknown"))
    claim_producer.send("claim-events", key=str(key), value=event)

claim_producer.flush()
claim_producer.close()

print(f"Produced {len(claim_events)} claim events to 'claim-events' topic")

# Verify device-events count via batch read
device_batch = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "device-events")
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load()
)
print(f"Device events in Kafka: {device_batch.count()}")

### Step 3.2: Streaming Anomaly Detection

We apply the same clinical threshold rules from Week 9, but now on a live stream:

- **Heart rate > 100 bpm** — Tachycardia alert
- **SpO2 < 92%** — Hypoxemia alert
- **Systolic BP > 160 mmHg** — Hypertension alert

Any event matching at least one rule is flagged as an anomaly and written to the `critical-alerts` Kafka topic for downstream consumption.

In [ ]:
# Read device events as a stream and detect anomalies in real time
device_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "device-events")
    .option("startingOffsets", "earliest")
    .load()
    .selectExpr("CAST(value AS STRING) as json_str", "timestamp as kafka_ts")
    .select(F.from_json(F.col("json_str"), device_schema).alias("data"), "kafka_ts")
    .select("data.*", "kafka_ts")
)

# Apply anomaly rules
anomalies = device_stream.withColumn(
    "anomaly_flags",
    F.array_remove(
        F.array(
            F.when(F.col("reading.heart_rate") > 100, F.lit("TACHYCARDIA")),
            F.when(F.col("reading.spo2") < 92, F.lit("HYPOXEMIA")),
            F.when(F.col("reading.bp_systolic") > 160, F.lit("HYPERTENSION")),
        ),
        None  # Remove nulls (non-matching rules)
    )
).filter(F.size("anomaly_flags") > 0)

# Select alert output columns
alerts = anomalies.select(
    "device_id",
    "patient_id",
    "hospital_id",
    "timestamp",
    "reading.heart_rate",
    "reading.spo2",
    "reading.bp_systolic",
    "anomaly_flags",
)

print("Anomaly detection stream schema:")
alerts.printSchema()

In [ ]:
# Write anomaly alerts to the console to observe them
alert_query = (
    alerts
    .writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", False)
    .option("numRows", 20)
    .trigger(processingTime="5 seconds")
    .start()
)

print("Real-time anomaly detection running...\n")
alert_query.awaitTermination(timeout=30)
alert_query.stop()
print("\nAnomaly detection query stopped.")

### Step 3.3: Windowed Escalation — Hospital-Level Alerts

Individual anomalies are useful, but a cluster of anomalies at the same hospital within a short time window suggests a systemic issue (e.g., a faulty device batch, an environmental problem, or an emerging patient crisis).

We group anomalies by `hospital_id` in **5-minute tumbling windows** and flag any hospital with more than 3 anomalies as needing escalation.

In [ ]:
# Re-read the stream for the escalation query
device_stream_2 = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "device-events")
    .option("startingOffsets", "earliest")
    .load()
    .selectExpr("CAST(value AS STRING) as json_str")
    .select(F.from_json(F.col("json_str"), device_schema).alias("data"))
    .select("data.*")
)

# Filter to anomalies, then window-aggregate by hospital
escalation = (
    device_stream_2
    .filter(
        (F.col("reading.heart_rate") > 100) |
        (F.col("reading.spo2") < 92) |
        (F.col("reading.bp_systolic") > 160)
    )
    .withColumn("event_time", F.to_timestamp("timestamp"))
    .groupBy(
        F.window("event_time", "5 minutes"),
        "hospital_id"
    )
    .agg(
        F.count("*").alias("anomaly_count"),
        F.collect_set("patient_id").alias("affected_patients"),
    )
    .withColumn(
        "escalation_needed",
        F.when(F.col("anomaly_count") > 3, F.lit("YES")).otherwise(F.lit("NO"))
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "hospital_id",
        "anomaly_count",
        "escalation_needed",
        F.size("affected_patients").alias("unique_patients"),
    )
)

escalation_query = (
    escalation
    .writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", False)
    .option("numRows", 30)
    .trigger(processingTime="5 seconds")
    .start()
)

print("Hospital escalation monitoring running...\n")
escalation_query.awaitTermination(timeout=30)
escalation_query.stop()
print("\nEscalation query stopped.")

### Step 3.4: Verify Results — Batch vs. Streaming Comparison

Let us read the device events as a batch and count anomalies to verify our streaming detection is consistent.

In [ ]:
# Batch verification: count anomalies using the same rules
batch_devices = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "device-events")
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load()
    .selectExpr("CAST(value AS STRING) as json_str")
    .select(F.from_json(F.col("json_str"), device_schema).alias("data"))
    .select("data.*")
)

total_events = batch_devices.count()
anomaly_count = batch_devices.filter(
    (F.col("reading.heart_rate") > 100) |
    (F.col("reading.spo2") < 92) |
    (F.col("reading.bp_systolic") > 160)
).count()

print(f"Total device events: {total_events}")
print(f"Anomalous events:    {anomaly_count} ({anomaly_count/total_events*100:.1f}%)")
print(f"Normal events:       {total_events - anomaly_count}")

print("\n--- Comparison ---")
print("In Week 9, we detected these same anomalies in a nightly batch job.")
print("In Week 10, we detect them within seconds of the event arriving in Kafka.")
print("The anomaly rules are IDENTICAL — only the execution model changed.")

---

## Part 4: GreenRoute — Live Fleet Tracking (Kappa Architecture)

GreenRoute uses the **Kappa Architecture** — there is no separate batch layer. All data flows through Kafka, and streaming queries provide both real-time insights and historical analysis (by replaying from Kafka or a downstream data lake).

In this section, we will:

1. Detect idle trucks from GPS telemetry in real time
2. Track delivery lifecycle events
3. Incorporate weather alerts for route planning

### Step 4.1: Load GPS Events into Kafka

In [ ]:
# Load GPS events and produce to Kafka
with open("/home/jovyan/data/greenroute/sample-gps-events.json") as f:
    gps_events = json.load(f)

gps_producer = KafkaProducer(
    bootstrap_servers="kafka:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: k.encode("utf-8") if k else None,
)

for event in gps_events:
    key = event.get("truck_id", "unknown")
    gps_producer.send("gps-stream", key=str(key), value=event)

gps_producer.flush()
gps_producer.close()

print(f"Produced {len(gps_events)} GPS events to 'gps-stream' topic")
if gps_events:
    print(f"\nSample event keys: {list(gps_events[0].keys())}")
    print(f"Sample event: {json.dumps(gps_events[0], indent=2)[:300]}")

### Step 4.2: Idle Truck Detection

A truck with an average speed below 2 mph over a 5-minute window is flagged as idle. Idle trucks waste fuel and delay deliveries — real-time detection enables dispatchers to intervene immediately.

In [ ]:
# Define the GPS event schema (adjust based on actual data)
from pyspark.sql.types import DoubleType

gps_schema = StructType([
    StructField("truck_id", StringType()),
    StructField("timestamp", StringType()),
    StructField("latitude", DoubleType()),
    StructField("longitude", DoubleType()),
    StructField("speed_mph", FloatType()),
    StructField("heading", FloatType()),
    StructField("fuel_level", FloatType()),
    StructField("engine_status", StringType()),
])

# Read GPS stream from Kafka
gps_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "gps-stream")
    .option("startingOffsets", "earliest")
    .load()
    .selectExpr("CAST(value AS STRING) as json_str")
    .select(F.from_json(F.col("json_str"), gps_schema).alias("data"))
    .select("data.*")
)

# Idle detection: 5-minute tumbling window, average speed < 2 mph
idle_detection = (
    gps_stream
    .withColumn("event_time", F.to_timestamp("timestamp"))
    .groupBy(
        F.window("event_time", "5 minutes"),
        "truck_id"
    )
    .agg(
        F.avg("speed_mph").alias("avg_speed"),
        F.count("*").alias("reading_count"),
        F.last("latitude").alias("last_lat"),
        F.last("longitude").alias("last_lon"),
    )
    .withColumn(
        "status",
        F.when(F.col("avg_speed") < 2.0, F.lit("IDLE"))
         .otherwise(F.lit("MOVING"))
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "truck_id",
        F.round("avg_speed", 1).alias("avg_speed_mph"),
        "reading_count",
        "status",
        F.round("last_lat", 4).alias("last_latitude"),
        F.round("last_lon", 4).alias("last_longitude"),
    )
)

idle_query = (
    idle_detection
    .writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", False)
    .option("numRows", 30)
    .trigger(processingTime="5 seconds")
    .start()
)

print("Idle truck detection running...\n")
idle_query.awaitTermination(timeout=30)
idle_query.stop()
print("\nIdle detection query stopped.")

In [ ]:
# Summary: count idle vs moving windows from batch read
gps_batch = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "gps-stream")
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load()
    .selectExpr("CAST(value AS STRING) as json_str")
    .select(F.from_json(F.col("json_str"), gps_schema).alias("data"))
    .select("data.*")
)

total_gps = gps_batch.count()
idle_count = gps_batch.filter(F.col("speed_mph") < 2.0).count()

print(f"Total GPS events: {total_gps}")
print(f"Low-speed events (< 2 mph): {idle_count} ({idle_count/max(total_gps,1)*100:.1f}%)")
print(f"\nUnique trucks: {gps_batch.select('truck_id').distinct().count()}")

### Step 4.3: Load Delivery Events

In [ ]:
# Produce delivery events to Kafka
with open("/home/jovyan/data/greenroute/sample-delivery-events.json") as f:
    delivery_events = json.load(f)

delivery_producer = KafkaProducer(
    bootstrap_servers="kafka:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: k.encode("utf-8") if k else None,
)

for event in delivery_events:
    key = event.get("delivery_id", event.get("truck_id", "unknown"))
    delivery_producer.send("delivery-events", key=str(key), value=event)

delivery_producer.flush()
delivery_producer.close()

print(f"Produced {len(delivery_events)} delivery events to 'delivery-events' topic")
if delivery_events:
    print(f"Sample event keys: {list(delivery_events[0].keys())}")

### Step 4.4: Delivery Lifecycle Tracking

Delivery events follow a lifecycle: `dispatched` -> `in_transit` -> `delivered` (or `failed`). By streaming these events, dispatchers see delivery progress in real time.

In [ ]:
# Define delivery event schema (adjust based on actual data)
delivery_schema = StructType([
    StructField("delivery_id", StringType()),
    StructField("truck_id", StringType()),
    StructField("status", StringType()),
    StructField("timestamp", StringType()),
    StructField("origin", StringType()),
    StructField("destination", StringType()),
    StructField("customer_id", StringType()),
    StructField("weight_lbs", FloatType()),
])

# Read delivery events as a stream
delivery_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "delivery-events")
    .option("startingOffsets", "earliest")
    .load()
    .selectExpr("CAST(value AS STRING) as json_str")
    .select(F.from_json(F.col("json_str"), delivery_schema).alias("data"))
    .select("data.*")
)

# Track delivery status distribution
delivery_status = (
    delivery_stream
    .groupBy("status")
    .agg(
        F.count("*").alias("count"),
        F.countDistinct("truck_id").alias("unique_trucks"),
    )
)

delivery_query = (
    delivery_status
    .writeStream
    .outputMode("complete")
    .format("console")
    .option("truncate", False)
    .trigger(processingTime="5 seconds")
    .start()
)

print("Delivery lifecycle tracking running...\n")
delivery_query.awaitTermination(timeout=20)
delivery_query.stop()
print("\nDelivery tracking query stopped.")

### Step 4.5: Weather Alerts

Weather alerts affect route planning. In a production Kappa architecture, weather alerts would flow into Kafka as a separate topic and be joined with GPS data to trigger rerouting decisions.

In [ ]:
# Load weather alerts to Kafka
with open("/home/jovyan/data/greenroute/sample-weather-alerts.json") as f:
    weather_alerts = json.load(f)

weather_producer = KafkaProducer(
    bootstrap_servers="kafka:9092",
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: k.encode("utf-8") if k else None,
)

for alert in weather_alerts:
    key = alert.get("alert_id", alert.get("region", "unknown"))
    weather_producer.send("weather-alerts", key=str(key), value=alert)

weather_producer.flush()
weather_producer.close()

print(f"Produced {len(weather_alerts)} weather alerts to 'weather-alerts' topic")

In [ ]:
# Read weather alerts as a batch to inspect the data
weather_batch = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "weather-alerts")
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load()
    .selectExpr("CAST(value AS STRING) as json_str")
)

# Parse and display
weather_df = weather_batch.select(
    F.from_json(F.col("json_str"), 
        StructType([
            StructField("alert_id", StringType()),
            StructField("alert_type", StringType()),
            StructField("severity", StringType()),
            StructField("region", StringType()),
            StructField("timestamp", StringType()),
            StructField("description", StringType()),
        ])
    ).alias("data")
).select("data.*")

print(f"Weather alerts loaded: {weather_df.count()}")
weather_df.show(10, truncate=False)

**How would weather join with GPS in production?**

In a full Kappa pipeline, you would use a **stream-stream join** or a **broadcast join** (if the weather dataset is small enough to fit in memory):

```python
# Stream-stream join with a time-based watermark
gps_with_weather = (
    gps_stream
    .withWatermark("event_time", "10 minutes")
    .join(
        weather_stream.withWatermark("alert_time", "10 minutes"),
        on=[gps_stream.region == weather_stream.region],
        how="left"
    )
)
```

Trucks entering a severe weather region would be flagged for rerouting. The watermark ensures Spark does not hold state indefinitely, releasing old events after the specified delay.

### Step 4.6: Architecture Reflection — Kappa in Practice

GreenRoute uses the **Kappa Architecture**: all data flows through Kafka as the single source of truth.

| Question | Kappa Answer |
|---|---|
| **Real-time fleet tracking?** | Read from GPS stream directly |
| **Historical trip analysis?** | Replay events from Kafka (7-day retention) or read from a downstream data lake |
| **Add a new metric?** | Deploy a new streaming consumer — it reads from the same Kafka topics |
| **Reprocess after a bug fix?** | Reset consumer offsets and replay from the beginning |

There is **no separate batch pipeline**. If you need historical analysis beyond Kafka's retention window, the streaming pipeline writes to a durable store (e.g., Parquet on HDFS or S3), which then serves as the long-term archive. But the processing logic lives in one place: the streaming layer.

**Contrast with HealthPulse (Lambda):** HealthPulse maintains a separate batch pipeline (Week 9) for comprehensive nightly analysis and a speed layer (this notebook) for real-time alerts. The trade-off is operational complexity (two codebases to maintain) in exchange for the batch layer's ability to do full historical recomputation with different tools and optimizations.

---

## Part 5: Reflection Questions

Consider the following questions. They will help you synthesize the concepts from this week's lecture and hands-on work:

1. **Batch vs. Streaming Code:** How does the code differ between batch and streaming Spark? What stays the same? Where are the only differences?

2. **Lambda vs. Kappa:** HealthPulse uses Lambda (batch + stream). GreenRoute uses Kappa (stream only). What factors drive this architectural choice? Consider data volume, latency requirements, complexity tolerance, and historical analysis needs.

3. **Kafka Retention Limits:** If Kafka retains 7 days of events, what happens when you need to analyze 6 months of GPS data? How would you architect the system to support both real-time tracking and long-term analytics?

4. **Rule Synchronization:** The anomaly rules (heart_rate > 100, spo2 < 92, bp_systolic > 160) are identical in batch (Week 9) and streaming (Week 10). In a production Lambda architecture, how would you ensure these rules stay synchronized? What happens if they diverge?

5. **Scaling Considerations:** Our lab uses a single Kafka broker and one Spark worker. What would change in a production deployment with millions of events per second? Think about partition counts, consumer groups, Spark executor configuration, and fault tolerance.

---

## Summary

In this companion notebook, you explored Apache Kafka and Spark Structured Streaming through two production-style business scenarios:

| | HealthPulse (Lambda) | GreenRoute (Kappa) |
|---|---|---|
| **Architecture** | Batch (Week 9) + Speed layer (Week 10) | Stream-only — all data through Kafka |
| **Week 9 contribution** | Schema harmonization, anomaly flagging in batch | GPS trip segmentation, delivery enrichment in batch |
| **Week 10 addition** | Real-time anomaly detection, hospital escalation alerts | Live idle truck detection, delivery lifecycle tracking, weather integration |
| **Kafka topics** | `device-events`, `critical-alerts` | `gps-stream`, `delivery-events`, `weather-alerts` |
| **Streaming patterns** | Filter + append, windowed aggregation | Windowed aggregation, status tracking |
| **Key insight** | Same rules, seconds instead of hours | Single codebase for real-time and historical |

**Key takeaways:**

1. **Kafka is a distributed log**, not a queue — consuming does not delete events, enabling replay and multiple independent consumers.
2. **Structured Streaming uses the same DataFrame API as batch** — `readStream`/`writeStream` replace `read`/`write`, but all transformations are identical.
3. **Windowed aggregations** are the foundation of streaming analytics — tumbling and sliding windows group events by time for real-time metrics.
4. **Lambda vs. Kappa** is an architectural trade-off between operational complexity and processing flexibility.
5. **The streaming layer does not replace the batch layer** (in Lambda) — it provides low-latency approximate results while batch provides comprehensive historical truth.

**Next week:** We will explore **data governance and pipeline orchestration** — how to manage data quality, lineage, and scheduling across the batch and streaming pipelines you have built.

In [ ]:
# Clean up: stop the SparkSession
spark.stop()
print("SparkSession stopped. You can now shut down the cluster with:")
print("  docker compose down")